# Approach 5: Hybrid Final Pipeline

This notebook combines the useful signals from previous approaches and adds post-processing learned from the observed errors:

- VAD is reliable for coarse speech activity, even when ASR is weak.
- WhisperX ASR helps on English and mixed English/Hindi files.
- WhisperX/pyannote diarization should be used for continuity, but may be unavailable or empty in some environments.
- Kannada-heavy ASR can degrade into repeated syllables, so the final system must detect low transcript quality and fall back to acoustic/structural segmentation.
- The assignment prior says each file contains exactly **two customer conversations**, so the final stage explicitly selects two conversation windows.

Approach 5 consumes the diagnostic CSVs exported by approach 4 and writes final hybrid predictions plus evaluation against `ground_truth.json` when available.


In [1]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        has_audio = (candidate / "audio").exists()
        has_repo_marker = (candidate / "ground_truth.json").exists() or (candidate / "approaches").exists() or (candidate / "context.md").exists()
        if has_audio and has_repo_marker:
            return candidate
        # If the notebook is launched from inside approaches/, the parent is the project root.
        if candidate.name == "approaches" and (candidate.parent / "audio").exists():
            return candidate.parent
    raise FileNotFoundError(
        "Could not find project root. Expected a folder containing audio/ and ground_truth.json or approaches/. "
        f"Current working directory was: {start}"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
APPROACH_4_DIR = PROJECT_ROOT / "approaches" / "approach_4_outputs"
OUTPUT_DIR = PROJECT_ROOT / "approaches" / "approach_5_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "expected_conversations": 2,
    "separator_gap_s": 50.0,
    "min_side_span_s": 60.0,
    "tail_trim_gap_s": 25.0,
    "tail_trim_max_speech_after_s": 25.0,
    "tail_trim_max_after_ratio": 0.18,
    "dense_split_fraction": 0.40,
    "dense_split_min_fraction": 0.25,
    "dense_split_max_fraction": 0.65,
    "small_gap_attach_s": 12.0,
    "low_asr_repetition_threshold": 0.42,
    "usable_latin_char_ratio": 0.55,
    "low_asr_min_speaker_rows": 2,
}

truth_path = PROJECT_ROOT / "ground_truth.json"
GROUND_TRUTH = json.loads(truth_path.read_text()) if truth_path.exists() else {}

audio_names = sorted(GROUND_TRUTH) if GROUND_TRUTH else sorted({p.name.replace("_speech_segments.csv", ".mp3") for p in APPROACH_4_DIR.glob("*_speech_segments.csv")})

print("Project root:", PROJECT_ROOT)
print("Approach 4 outputs:", APPROACH_4_DIR)
print("Approach 5 outputs:", OUTPUT_DIR)
print("Files:", audio_names)

import sys
sys.path.append(str(PROJECT_ROOT / "approaches"))
from output_utils import export_uniform_outputs, export_combined_outputs


Project root: /teamspace/studios/this_studio
Approach 4 outputs: /teamspace/studios/this_studio/approaches/approach_4_outputs
Approach 5 outputs: /teamspace/studios/this_studio/approaches/approach_5_outputs
Files: ['Sample1KN.mp3', 'Sample2EN.mp3', 'sample3KN.mp3']


In [3]:
def stem_for_audio(audio_name: str) -> str:
    return Path(audio_name).stem


def read_csv_optional(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 1:
        return pd.read_csv(path)
    return pd.DataFrame()


def read_approach4_csv(stem: str, filename: str) -> pd.DataFrame:
    main_path = APPROACH_4_DIR / filename
    debug_path = APPROACH_4_DIR / "_debug" / filename
    if main_path.exists() and main_path.stat().st_size > 1:
        return pd.read_csv(main_path)
    if debug_path.exists() and debug_path.stat().st_size > 1:
        return pd.read_csv(debug_path)
    return pd.DataFrame()


def format_time(seconds: float) -> str:
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes:02d}:{secs:06.3f}"


def load_approach4_artifacts(audio_name: str) -> dict:
    stem = stem_for_audio(audio_name)
    return {
        "audio_name": audio_name,
        "stem": stem,
        "speech": read_approach4_csv(stem, f"{stem}_speech_segments.csv"),
        "candidates": read_approach4_csv(stem, f"{stem}_conversation_candidates.csv"),
        "asr": read_approach4_csv(stem, f"{stem}_whisperx_asr_segments.csv"),
        "speaker_turns": read_approach4_csv(stem, f"{stem}_whisperx_speaker_turns.csv"),
        "candidate_summary": read_approach4_csv(stem, f"{stem}_whisperx_candidate_summary.csv"),
        "merged_candidate_summary": read_approach4_csv(stem, f"{stem}_whisperx_merged_candidate_summary.csv"),
        "approach4_final": read_approach4_csv(stem, f"{stem}_final_two_conversations.csv"),
    }


def add_time_columns(rows: list[dict]) -> list[dict]:
    for row in rows:
        row["start_hms"] = format_time(float(row["start"]))
        row["end_hms"] = format_time(float(row["end"]))
    return rows


In [4]:
def token_repetition_ratio(text: str) -> float:
    tokens = re.findall(r"\w+", str(text).lower())
    if not tokens:
        return 1.0
    return 1.0 - (len(set(tokens)) / len(tokens))


def repeated_character_ratio(text: str) -> float:
    text = re.sub(r"\s+", "", str(text))
    if len(text) < 2:
        return 0.0
    repeated = sum(1 for left, right in zip(text, text[1:]) if left == right)
    return repeated / max(len(text) - 1, 1)


def estimate_asr_quality(artifacts: dict) -> dict:
    asr = artifacts["asr"]
    turns = artifacts["speaker_turns"]
    if asr.empty:
        return {"asr_quality": "missing", "repetition_ratio": 1.0, "char_repeat_ratio": 1.0, "latin_char_ratio": 0.0, "speaker_rows": len(turns)}

    text = " ".join(asr.get("text", pd.Series(dtype=str)).fillna("").astype(str).tolist())
    repetition = token_repetition_ratio(text)
    char_repeat = repeated_character_ratio(text)
    chars = [char for char in text if not char.isspace()]
    latin_char_ratio = sum(1 for char in chars if "a" <= char.lower() <= "z") / max(len(chars), 1)
    speaker_rows = len(turns)
    has_speaker_labels = speaker_rows >= CONFIG["low_asr_min_speaker_rows"]
    transcript_usable = latin_char_ratio >= CONFIG["usable_latin_char_ratio"] or has_speaker_labels
    low_quality = not transcript_usable

    return {
        "asr_quality": "low" if low_quality else "usable",
        "repetition_ratio": repetition,
        "char_repeat_ratio": char_repeat,
        "latin_char_ratio": latin_char_ratio,
        "speaker_rows": speaker_rows,
    }


def speech_gaps(speech: pd.DataFrame) -> list[dict]:
    if speech.empty:
        return []
    rows = speech.sort_values("start").to_dict("records")
    gaps = []
    for left, right in zip(rows, rows[1:]):
        gap = float(right["start"]) - float(left["end"])
        if gap > 0:
            gaps.append({"prev_end": float(left["end"]), "next_start": float(right["start"]), "gap": gap})
    return gaps


def speech_duration_between(speech: pd.DataFrame, start: float, end: float) -> float:
    if speech.empty:
        return 0.0
    total = 0.0
    for row in speech.to_dict("records"):
        overlap = max(0.0, min(float(row["end"]), end) - max(float(row["start"]), start))
        total += overlap
    return total


def active_bounds(artifacts: dict) -> tuple[float, float]:
    starts, ends = [], []
    for key in ["speech", "asr"]:
        df = artifacts[key]
        if not df.empty:
            starts.append(float(df["start"].min()))
            ends.append(float(df["end"].max()))
    if not starts:
        return 0.0, 0.0
    return min(starts), max(ends)


In [5]:
def choose_separator_gap(speech: pd.DataFrame, active_start: float, active_end: float, cfg: dict):
    total_span = active_end - active_start
    candidates = []
    for gap in speech_gaps(speech):
        if gap["gap"] < cfg["separator_gap_s"]:
            continue
        left_span = gap["prev_end"] - active_start
        right_span = active_end - gap["next_start"]
        if left_span < cfg["min_side_span_s"] or right_span < cfg["min_side_span_s"]:
            continue
        ratio = (gap["prev_end"] - active_start) / max(total_span, 1e-6)
        balance = min(left_span, right_span) / max(left_span, right_span)
        position_bonus = 45.0 if ratio >= 0.45 else 0.0
        score = gap["gap"] + 60.0 * balance + position_bonus
        candidates.append({**gap, "left_span": left_span, "right_span": right_span, "ratio": ratio, "score": score})
    if not candidates:
        return None
    return max(candidates, key=lambda row: row["score"])


def last_substantive_candidate_end(summary: pd.DataFrame, interval_start: float, interval_end: float):
    if summary.empty:
        return None
    df = summary[(summary["start"] >= interval_start) & (summary["end"] <= interval_end)].copy()
    if df.empty:
        return None
    if "interaction_like" in df.columns:
        strong = df[df["interaction_like"].astype(str).str.lower().isin(["true", "1"])]
        if not strong.empty:
            end = float(strong["end"].max())
            # Attach one short nearby trailing candidate because boundaries often include a short closing utterance.
            tail = df[(df["start"] > end) & (df["start"] - end <= CONFIG["small_gap_attach_s"])]
            if not tail.empty:
                end = float(tail["end"].max())
            return end
    scored = df[df.get("conversation_score", 0) > 0] if "conversation_score" in df.columns else df
    if not scored.empty:
        return float(scored["end"].max())
    return None


def trim_interval_tail(interval: dict, speech: pd.DataFrame, summary: pd.DataFrame, cfg: dict, use_summary_trim: bool):
    start, end = interval["start"], interval["end"]
    if use_summary_trim:
        substantive_end = last_substantive_candidate_end(summary, start, end)
        if substantive_end is not None and substantive_end > start:
            end = min(end, substantive_end)

    total_speech = speech_duration_between(speech, start, end)
    for gap in speech_gaps(speech):
        if gap["prev_end"] <= start or gap["next_start"] >= end:
            continue
        if gap["gap"] < cfg["tail_trim_gap_s"]:
            continue
        if gap["prev_end"] < start + 0.55 * (end - start):
            continue
        speech_after = speech_duration_between(speech, gap["next_start"], end)
        if speech_after <= cfg["tail_trim_max_speech_after_s"] or speech_after / max(total_speech, 1e-6) <= cfg["tail_trim_max_after_ratio"]:
            end = gap["prev_end"]
            break

    return {**interval, "end": end, "duration": end - start}


def dense_two_way_split(artifacts: dict, active_start: float, active_end: float, cfg: dict):
    asr = artifacts["asr"]
    target = active_start + cfg["dense_split_fraction"] * (active_end - active_start)
    min_t = active_start + cfg["dense_split_min_fraction"] * (active_end - active_start)
    max_t = active_start + cfg["dense_split_max_fraction"] * (active_end - active_start)

    boundaries = []
    if not asr.empty:
        rows = asr.sort_values("start").to_dict("records")
        for left, right in zip(rows, rows[1:]):
            prev_end = float(left["end"])
            next_start = float(right["start"])
            boundary = (prev_end + next_start) / 2.0
            if min_t <= boundary <= max_t:
                boundaries.append({"prev_end": prev_end, "next_start": next_start, "boundary": boundary, "distance": abs(boundary - target)})

    if boundaries:
        split = min(boundaries, key=lambda row: row["distance"])
        return [
            {"customer_index": 1, "start": active_start, "end": split["prev_end"], "method": "dense_asr_boundary"},
            {"customer_index": 2, "start": split["next_start"], "end": active_end, "method": "dense_asr_boundary"},
        ]

    split = target
    return [
        {"customer_index": 1, "start": active_start, "end": split, "method": "dense_fraction_fallback"},
        {"customer_index": 2, "start": split, "end": active_end, "method": "dense_fraction_fallback"},
    ]


In [6]:
def hybrid_predict_two_conversations(audio_name: str, cfg: dict) -> tuple[list[dict], dict]:
    artifacts = load_approach4_artifacts(audio_name)
    quality = estimate_asr_quality(artifacts)
    speech = artifacts["speech"]
    summary = artifacts["candidate_summary"]
    active_start, active_end = active_bounds(artifacts)
    separator = choose_separator_gap(speech, active_start, active_end, cfg)

    if separator is None:
        intervals = dense_two_way_split(artifacts, active_start, active_end, cfg)
        strategy = "dense_split_no_major_separator"
    else:
        first = {"customer_index": 1, "start": active_start, "end": separator["prev_end"], "method": "major_separator"}
        second = {"customer_index": 2, "start": separator["next_start"], "end": active_end, "method": "major_separator"}
        # Use ASR summary trim when transcript looks usable; use acoustic tail trim when ASR is low quality.
        first = trim_interval_tail(first, speech, summary, cfg, use_summary_trim=(quality["asr_quality"] == "usable"))
        if quality["asr_quality"] == "low":
            first = trim_interval_tail(first, speech, summary, cfg, use_summary_trim=False)
        intervals = [first, second]
        strategy = "major_separator_with_quality_aware_trimming"

    rows = []
    for row in intervals[: cfg["expected_conversations"]]:
        start = max(0.0, float(row["start"]))
        end = max(start, float(row["end"]))
        rows.append(
            {
                "audio_name": audio_name,
                "customer_index": int(row["customer_index"]),
                "start": start,
                "end": end,
                "duration": end - start,
                "method": row.get("method", strategy),
                "strategy": strategy,
                "asr_quality": quality["asr_quality"],
                "asr_repetition_ratio": quality["repetition_ratio"],
                "speaker_rows": quality["speaker_rows"],
            }
        )

    rows = add_time_columns(rows)
    diagnostics = {"audio_name": audio_name, "strategy": strategy, **quality, "separator": separator}
    return rows, diagnostics


def interval_iou(pred_start: float, pred_end: float, gt_start: float, gt_end: float) -> float:
    inter = max(0.0, min(pred_end, gt_end) - max(pred_start, gt_start))
    union = max(pred_end, gt_end) - min(pred_start, gt_start)
    return inter / union if union > 0 else 0.0


def evaluate_predictions(rows: list[dict], ground_truth: dict) -> list[dict]:
    eval_rows = []
    for row in rows:
        audio_name = row["audio_name"]
        idx = int(row["customer_index"])
        gt = ground_truth.get(audio_name, {}).get(f"Conversation {idx}")
        if not gt:
            continue
        eval_rows.append(
            {
                "audio_name": audio_name,
                "customer_index": idx,
                "pred_start": row["start"],
                "pred_end": row["end"],
                "gt_start": gt["start"],
                "gt_end": gt["end"],
                "start_error_s": row["start"] - gt["start"],
                "end_error_s": row["end"] - gt["end"],
                "abs_start_error_s": abs(row["start"] - gt["start"]),
                "abs_end_error_s": abs(row["end"] - gt["end"]),
                "iou": interval_iou(row["start"], row["end"], gt["start"], gt["end"]),
                "method": row["method"],
                "strategy": row["strategy"],
                "asr_quality": row["asr_quality"],
            }
        )
    return eval_rows


In [7]:
all_predictions = []
all_diagnostics = []

for audio_name in audio_names:
    rows, diagnostics = hybrid_predict_two_conversations(audio_name, CONFIG)
    all_predictions.extend(rows)
    all_diagnostics.append(diagnostics)

pred_df = pd.DataFrame(all_predictions)
diag_df = pd.DataFrame(all_diagnostics)

exported_outputs = []
for audio_name, group in pred_df.groupby("audio_name", sort=False):
    artifacts = load_approach4_artifacts(audio_name)
    exported_outputs.append(
        export_uniform_outputs(
            output_dir=OUTPUT_DIR,
            audio_name=audio_name,
            approach_name="approach_5_hybrid",
            conversation_candidates=artifacts["candidates"],
            final_two=group,
            ground_truth=GROUND_TRUTH,
            final_selection_method="hybrid_quality_aware",
        )
    )

combined_outputs = export_combined_outputs(OUTPUT_DIR, exported_outputs)
pred_df = combined_outputs["all_files_final_two_conversations"]
eval_df = combined_outputs["evaluation_against_ground_truth"]
diag_df.to_csv(OUTPUT_DIR / "hybrid_diagnostics.csv", index=False)

display(pred_df)
display(diag_df[["audio_name", "strategy", "asr_quality", "latin_char_ratio", "repetition_ratio", "speaker_rows"]])
if not eval_df.empty:
    display(eval_df)


,audio_name,customer_index,start,end,duration,start_hms,end_hms,method,score,source_conversation_ids
0,Sample1KN.mp3,1,0.031,268.023,267.992,00:00.031,04:28.023,dense_asr_boundary,<NA>,<NA>
1,Sample1KN.mp3,2,268.023,661.818,393.795,04:28.023,11:01.818,dense_asr_boundary,<NA>,<NA>
2,Sample2EN.mp3,1,18.183,379.590,361.407,00:18.183,06:19.590,dense_asr_boundary,<NA>,<NA>
3,Sample2EN.mp3,2,379.610,921.340,541.730,06:19.610,15:21.340,dense_asr_boundary,<NA>,<NA>
4,sample3KN.mp3,1,0.031,146.894,146.863,00:00.031,02:26.894,dense_asr_boundary,<NA>,<NA>
5,sample3KN.mp3,2,148.379,362.472,214.093,02:28.379,06:02.472,dense_asr_boundary,<NA>,<NA>


,audio_name,strategy,asr_quality,latin_char_ratio,repetition_ratio,speaker_rows
0,Sample1KN.mp3,dense_split_no_major_separator,low,0.146522,0.910336,0
1,Sample2EN.mp3,dense_split_no_major_separator,usable,0.879495,0.771344,0
2,sample3KN.mp3,dense_split_no_major_separator,low,0.000000,0.932735,0


,approach,audio_name,customer_index,pred_start,pred_end,gt_start,gt_end,start_error_s,end_error_s,abs_start_error_s,abs_end_error_s,iou
0,approach_5_hybrid,Sample1KN.mp3,1,0.031,268.023,16.0,390.0,-15.969,-121.977,15.969,121.977,0.646264
1,approach_5_hybrid,Sample1KN.mp3,2,268.023,661.818,520.0,660.0,-251.977,1.818,251.977,1.818,0.355515
2,approach_5_hybrid,Sample2EN.mp3,1,18.183,379.590,18.0,390.0,0.183,-10.410,0.183,10.410,0.971524
3,approach_5_hybrid,Sample2EN.mp3,2,379.610,921.340,525.0,921.0,-145.390,0.340,145.390,0.340,0.730991
4,approach_5_hybrid,sample3KN.mp3,1,0.031,146.894,0.0,142.0,0.031,4.894,0.031,4.894,0.966472
5,approach_5_hybrid,sample3KN.mp3,2,148.379,362.472,148.0,362.0,0.379,0.472,0.379,0.472,0.996032


In [8]:
evaluation_summary_df = combined_outputs["evaluation_summary"]
if not evaluation_summary_df.empty:
    display(evaluation_summary_df)


,approach,audio_name,mean_abs_start_error_s,mean_abs_end_error_s,mean_iou
0,approach_5_hybrid,Sample1KN.mp3,133.9730,61.8975,0.500890
1,approach_5_hybrid,Sample2EN.mp3,72.7865,5.3750,0.851258
2,approach_5_hybrid,sample3KN.mp3,0.2050,2.6830,0.981252


## Standard Outputs

Each approach now writes the same comparable output structure. For each audio file:

- `*_conversation_candidates.csv`: candidate conversation blocks generated by this approach.
- `*_final_two_conversations.csv`: the final predicted two customer conversation windows.
- `*_tagged_conversation_candidates.csv`: the candidate blocks annotated with whether they contributed to customer 1, customer 2, or were not selected.

At the approach folder level:

- `all_files_final_two_conversations.csv`: final predictions across all audio files.
- `evaluation_against_ground_truth.csv`: per-conversation boundary errors and IoU when `ground_truth.json` is available.
- `evaluation_summary.csv`: per-file mean absolute start error, mean absolute end error, and mean IoU.
